In [1]:
!pip install -q "vllm==0.6.6.post1"
!pip install -q "transformers==4.46.3"
!pip install -q "torch==2.5.1"
!pip uninstall -y torchaudio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/88

In [5]:
MODEL = "microsoft/Phi-3-mini-4k-instruct"
NUM_PROMPTS = 20
MAX_NEW_TOKENS = 64
prompts_base = [
    "Explain gradient descent in simple terms.",
    "Summarize the French Revolution in two sentences.",
    "What's the difference between TCP and UDP?",
    "Describe how a binary search tree works.",
    "Write a haiku about autumn.",
]
prompts = [prompts_base[i % len(prompts_base)] for i in range(NUM_PROMPTS)]

In [1]:
%pip install -U \
    torch==2.5.1 \
    transformers==4.46.3 \
    accelerate \
    sentencepiece \
    safetensors

Sequential Hugging Face Baseline Benchmark

In [6]:
import torch, time, json
from transformers import AutoModelForCausalLM, AutoTokenizer



tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")
model.eval()

latencies = []
total_tokens = 0
start_all = time.perf_counter()

for i, p in enumerate(prompts):
    inputs = tokenizer(p, return_tensors="pt").to("cuda")
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    latencies.append(t1 - t0)
    total_tokens += out.shape[1] - inputs["input_ids"].shape[1]
    print(f"[{i+1}/{NUM_PROMPTS}] {t1-t0:.2f}s")

total_time = time.perf_counter() - start_all
latencies.sort()
baseline_results = {
    "throughput_req_per_sec": NUM_PROMPTS / total_time,
    "throughput_tok_per_sec": total_tokens / total_time,
    "avg_latency": sum(latencies) / len(latencies),
    "p50_latency": latencies[int(len(latencies)*0.5)],
    "p99_latency": latencies[-1],
}
print(json.dumps(baseline_results, indent=2))

# free memory before starting vLLM
del model
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[1/20] 2.61s
[2/20] 2.61s
[3/20] 2.60s
[4/20] 3.31s
[5/20] 2.62s
[6/20] 2.59s
[7/20] 2.67s
[8/20] 2.79s
[9/20] 3.07s
[10/20] 2.61s
[11/20] 2.61s
[12/20] 2.64s
[13/20] 3.24s
[14/20] 2.58s
[15/20] 2.61s
[16/20] 2.58s
[17/20] 2.82s
[18/20] 3.18s
[19/20] 2.74s
[20/20] 2.95s
{
  "throughput_req_per_sec": 0.3603679186067026,
  "throughput_tok_per_sec": 23.063546790828966,
  "avg_latency": 2.7728342490999807,
  "p50_latency": 2.6437028939999436,
  "p99_latency": 3.305656850999867
}


Setting up vLLM Server

In [ ]:
import subprocess, time, requests
MODEL = "microsoft/Phi-3-mini-4k-instruct"
NUM_PROMPTS = 20
MAX_NEW_TOKENS = 64
vllm_proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--port", "8000",
     "--gpu-memory-utilization", "0.7",
     "--dtype", "half"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

for _ in range(90):
    try:
        r = requests.get("http://localhost:8000/health")
        if r.status_code == 200:
            print("vLLM server ready.")
            break
    except Exception:
        pass
    time.sleep(5)
else:
    print("Not ready yet — checking logs:")
    print(vllm_proc.stdout.read())

vLLM server ready.


In [ ]:
print(vllm_proc.stdout.read())

vLLM Throughput and Latency Benchmark

In [ ]:
import asyncio, aiohttp, nest_asyncio, time, json
nest_asyncio.apply()

CONCURRENCY = 8

async def send(session, sem, prompt):
    async with sem:
        payload = {"model": MODEL, "messages": [{"role": "user", "content": prompt}],
                   "max_tokens": MAX_NEW_TOKENS, "temperature": 0.0}
        t0 = time.perf_counter()
        async with session.post("http://localhost:8000/v1/chat/completions", json=payload) as resp:
            data = await resp.json()
        t1 = time.perf_counter()
        toks = data.get("usage", {}).get("completion_tokens", 0)
        return t1 - t0, toks

async def run():
    sem = asyncio.Semaphore(CONCURRENCY)
    async with aiohttp.ClientSession() as session:
        start = time.perf_counter()
        results = await asyncio.gather(*[send(session, sem, p) for p in prompts])
        total_time = time.perf_counter() - start
    lat = sorted(r[0] for r in results)
    toks = sum(r[1] for r in results)
    return {
        "throughput_req_per_sec": NUM_PROMPTS / total_time,
        "throughput_tok_per_sec": toks / total_time,
        "avg_latency": sum(lat) / len(lat),
        "p50_latency": lat[int(len(lat)*0.5)],
        "p99_latency": lat[-1],
        "concurrency": CONCURRENCY,
    }

vllm_results = asyncio.run(run())
print(json.dumps(vllm_results, indent=2))

{
  "throughput_req_per_sec": 0.8764203675464305,
  "throughput_tok_per_sec": 50.30652909716511,
  "avg_latency": 2.281683325749964,
  "p50_latency": 2.5557909710000786,
  "p99_latency": 2.6295785479996994,
  "concurrency": 2
}


In [ ]:
speedup = vllm_results["throughput_req_per_sec"] / baseline_results["throughput_req_per_sec"]
print(f"Baseline: {baseline_results['throughput_req_per_sec']:.2f} req/sec")
print(f"vLLM:     {vllm_results['throughput_req_per_sec']:.2f} req/sec")
print(f"Speedup:  {speedup:.2f}x")

In [ ]:
import requests
r = requests.get("http://localhost:8000/metrics")
print(r.text[:1000])

# HELP vllm:iteration_tokens_total Histogram of number of tokens per engine_step.
# TYPE vllm:iteration_tokens_total histogram
vllm:iteration_tokens_total_sum{model_name="microsoft/Phi-3-mini-4k-instruct"} 0.0
vllm:iteration_tokens_total_bucket{le="1.0",model_name="microsoft/Phi-3-mini-4k-instruct"} 26.0
vllm:iteration_tokens_total_bucket{le="2.0",model_name="microsoft/Phi-3-mini-4k-instruct"} 26.0
vllm:iteration_tokens_total_bucket{le="4.0",model_name="microsoft/Phi-3-mini-4k-instruct"} 26.0
vllm:iteration_tokens_total_bucket{le="8.0",model_name="microsoft/Phi-3-mini-4k-instruct"} 26.0
vllm:iteration_tokens_total_bucket{le="16.0",model_name="microsoft/Phi-3-mini-4k-instruct"} 26.0
vllm:iteration_tokens_total_bucket{le="24.0",model_name="microsoft/Phi-3-mini-4k-instruct"} 26.0
vllm:iteration_tokens_total_bucket{le="32.0",model_name="microsoft/Phi-3-mini-4k-instruct"} 26.0
vllm:iteration_tokens_total_bucket{le="40.0",model_name="microsoft/Phi-3-mini-4k-instruct"} 26.0
vllm:iteration_tok

In [ ]:
!wget -q https://github.com/prometheus/prometheus/releases/download/v2.53.0/prometheus-2.53.0.linux-amd64.tar.gz
!tar xzf prometheus-2.53.0.linux-amd64.tar.gz

Setting up monitoring using Grafana and prometheus

In [ ]:
prometheus_config = """
global:
  scrape_interval: 5s
scrape_configs:
  - job_name: 'vllm'
    static_configs:
      - targets: ['localhost:8000']
"""
with open("prometheus-2.53.0.linux-amd64/prometheus.yml", "w") as f:
    f.write(prometheus_config)

In [ ]:
import subprocess
prom_proc = subprocess.Popen(
    ["./prometheus", "--config.file=prometheus.yml"],
    cwd="prometheus-2.53.0.linux-amd64",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

In [ ]:
!wget -q https://dl.grafana.com/oss/release/grafana-11.1.0.linux-amd64.tar.gz
!tar xzf grafana-11.1.0.linux-amd64.tar.gz

In [ ]:
grafana_proc = subprocess.Popen(
    ["./bin/grafana-server"],
    cwd="grafana-v11.1.0",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok

# sign up free at ngrok.com, get your authtoken, paste it below
ngrok.set_auth_token("3GxzKOkNxVzFbA8L7SzXBpBEYG6_7rHf7NPzAL9YbbqTBSGwe")
public_url = ngrok.connect(3000)
print("Grafana URL:", public_url)

Grafana URL: NgrokTunnel: "https://chloride-handcraft-overfeed.ngrok-free.dev" -> "http://localhost:3000"


In [ ]:
import fcntl, os
fd = grafana_proc.stdout.fileno()
fl = fcntl.fcntl(fd, fcntl.F_GETFL)
fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)
try:
    print(grafana_proc.stdout.read())
except Exception as e:
    print("(nothing new)", e)

logger=migrator t=2026-07-24T21:17:59.332934767Z level=info msg="Migration successfully executed" id="Add non-unique index alert_rule_tag_alert_id" duration=891.933µs
logger=migrator t=2026-07-24T21:21:23.726977165Z level=info msg="Executing migration" id="Drop old annotation table v4"
logger=migrator t=2026-07-24T21:21:23.727118563Z level=info msg="Migration successfully executed" id="Drop old annotation table v4" duration=147.635µs
logger=migrator t=2026-07-24T21:21:23.734686256Z level=info msg="Executing migration" id="create annotation table v5"
logger=migrator t=2026-07-24T21:21:23.735725086Z level=info msg="Migration successfully executed" id="create annotation table v5" duration=1.037914ms
logger=migrator t=2026-07-24T21:21:23.745356357Z level=info msg="Executing migration" id="add index annotation 0 v3"
logger=migrator t=2026-07-24T21:21:23.746336146Z level=info msg="Migration successfully executed" id="add index annotation 0 v3" duration=978.607µs
logger=migrator t=2026-07-24T